# TP3 - Regresión, análisis multivariante y métodos de remuestreo aplicados a datos de dinámica molecular

## Apunte de clase:
Spoiler: NO hay independencia entre los datos
¿Qué pasa con el tunel a lo largo del tiempo?
Teoricamente si es un polo de un canal que se esta abriendo es probable que el radio sea mas grande.
¿Qué variable es mas importante? El radio del polo, por ejemplo.
Hacer analisis de correlació o de clasificación
Cómo se distribuye el valor del radio a lo largo de todas las muestras.
Este problema es mas complejo que solo el radio, no alcanza con eso. No se va a encontrar diferencia entre los grupos. Eso no significa nada, pero no se puede determinar si el canal está abierto. Es un problema de analisis estadistico.

Ojo: Si un p-value sa bien, sospechar

Otro problema: hydrophobicity y hydropathy, quiza no deberían ir juntos
No hay faltantes
No hay outl.

Es importante notar que hay 3 estados, el final es cuando se rompió

Parte III:
Hasta acá estamos entendiendo de qué trata
Una correlación no es solo mirar que si x aumenta entonces y también. Sino que hay asociado un modelo matematico que dice como van a correlacionar. Se corrobora si siguen una linea (tambiepn puede haber log y exp)

Heatmap es un resumen de todas las variables. Se resumen todas las regresiones en un solo gráfico. Al verlo se puede deducir qué variables están relacionadas. Luego si vemos que hay correlación entre dos columnas, hay que sacar una, porque la que saquemos se va a explicar por medio de la otra.
Longitud con tortuosidad estan correlacionados porque mientras mas longitud hay mas tortuosidad. Su coeficiente es bastante alto.

Como son variables numericas se hace regresión lineal. Hay correlación de pearson y spearman. Para perseon hay que tener distribución normal. Pearson no es lo adecuado. Hay que usar Spearman. Es una trampa del tp.

Pensar en Pearson: Medir la distancia entre cada punto y la recta mas probable que explique la relación entre los dos (la raiz de los cuadrados).

Hay q usar el coeficiente de correlación (cuanto se acercan a una recta esos datos) y el p-valor (que indica la significacncia de cuanto explican esos datos a ese coeficiente en particular). Si el ceoficiente de correlación es bueno pero el p-value es malo, entonces no correlacionan y visceversa.


Parte IV:
Para no mirar todas las variables solo se piden alguas específicas.
Uno no es efecto del otro.

¿Como puedo ver si hay una relación causa efecto? Ver si tiene sentido, encontrar el mecanismo. Se puede hallar la correlación. Si al modificar una, también cambia la otra (pero no alcanza con esto último). Podrían estar vinculados por una misma variable pero ser independientes, por ejemplo comer helado y ataques de tiburones (conectados por la temperatura)

Parte V:



In [1]:
import pandas as pd

df = pd.read_csv("results.csv")

print(df.head())
print(df.shape)
print(df.info())
print(df.describe())

   frame  recipe_id   length      dz  tortuosity  bottleneck_radius  \
0      0          0  162.692  86.179    1.887838              0.349   
1      1          0  181.193  86.928    2.084403              0.632   
2      2          0  180.389  93.018    1.939291              0.303   
3      3          0  184.363  83.650    2.203981              0.008   
4      4          0  139.483  79.060    1.764268              0.541   

   bottleneck_free_radius  bottleneck_bradius  n_layers  charge  ionizable  \
0                   0.410               0.349       144      -3          9   
1                   0.865               0.632       144      -1         11   
2                   0.395               0.303       129       1         13   
3                  -0.162               0.008       154       1         13   
4                   0.559               0.541       126       1          7   

   num_positives  num_negatives  hydrophobicity  hydropathy  polarity  logp  \
0              3         

## 4

Con la información del enunciado, podemos decir que la variabilidad del túnel puede estudiarse desde dos grandes dimensiones:

1. Características estructurales

Tenemos variables que describen la geometría del túnel:

length → longitud.
dz → extensión en el eje z.
tortuosity → qué tan curvado/tortuoso es.
bottleneck_radius → radio del cuello de botella.
bottleneck_free_radius → radio libre.
bottleneck_bradius → otra medida del radio del cuello.
n_layers → cantidad de capas/residuos involucrados.

Estas variables podrían explicar parte de la variabilidad estructural del túnel.

Por ejemplo, podríamos encontrar que cuando el bottleneck_radius aumenta, también aumenta length, o que determinados valores del radio del cuello están asociados con cambios en la tortuosidad.

Pero todavía no sabemos si esas relaciones existen. Eso lo vamos a determinar con correlaciones y regresiones.


2. Características fisicoquímicas

También tenemos variables relacionadas con las propiedades químicas del túnel:

charge
ionizable
num_positives
num_negatives
hydrophobicity
hydropathy
polarity
logp
logd
logs
mutability

Estas variables podrían estar asociadas con cambios en las características estructurales.

Por ejemplo, una pregunta interesante sería:

¿Los cambios en el radio del cuello de botella están asociados con cambios en la carga, polaridad o hidrofobicidad del túnel?

Nuevamente, no podemos afirmarlo todavía.

### Respuesta temporal:
A partir de la descripción del conjunto de datos, la variabilidad del túnel de TMEM63A puede estudiarse considerando tanto características estructurales como fisicoquímicas. Las variables geométricas, como la longitud, la tortuosidad y los diferentes radios del cuello de botella, permiten caracterizar cambios en la configuración del túnel, mientras que variables como la carga, hidrofobicidad, polaridad y cantidad de residuos ionizables permiten estudiar sus propiedades fisicoquímicas.

El análisis estadístico permitirá determinar cuáles de estas variables están asociadas entre sí y cuáles aportan información diferente. Además, mediante técnicas multivariantes como PCA y clustering se podrá investigar si los frames de la trayectoria pueden agruparse en diferentes configuraciones o estados del túnel. Sin embargo, estas conclusiones no pueden establecerse únicamente a partir de la descripción de las variables y deberán ser verificadas mediante los análisis realizados a lo largo del trabajo.

In [1]:
import pandas as pd

df = pd.read_csv("results.csv")

print(df.shape)
print(df.dtypes)
print(df.isna().sum())

(1055, 20)
frame                       int64
recipe_id                   int64
length                    float64
dz                        float64
tortuosity                float64
bottleneck_radius         float64
bottleneck_free_radius    float64
bottleneck_bradius        float64
n_layers                    int64
charge                      int64
ionizable                   int64
num_positives               int64
num_negatives               int64
hydrophobicity            float64
hydropathy                float64
polarity                  float64
logp                      float64
logd                      float64
logs                      float64
mutability                  int64
dtype: object
frame                     0
recipe_id                 0
length                    0
dz                        0
tortuosity                0
bottleneck_radius         0
bottleneck_free_radius    0
bottleneck_bradius        0
n_layers                  0
charge                    0
ionizable      

In [2]:
print(df.head())

   frame  recipe_id   length      dz  tortuosity  bottleneck_radius  \
0      0          0  162.692  86.179    1.887838              0.349   
1      1          0  181.193  86.928    2.084403              0.632   
2      2          0  180.389  93.018    1.939291              0.303   
3      3          0  184.363  83.650    2.203981              0.008   
4      4          0  139.483  79.060    1.764268              0.541   

   bottleneck_free_radius  bottleneck_bradius  n_layers  charge  ionizable  \
0                   0.410               0.349       144      -3          9   
1                   0.865               0.632       144      -1         11   
2                   0.395               0.303       129       1         13   
3                  -0.162               0.008       154       1         13   
4                   0.559               0.541       126       1          7   

   num_positives  num_negatives  hydrophobicity  hydropathy  polarity  logp  \
0              3         

In [3]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 1055 entries, 0 to 1054
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   frame                   1055 non-null   int64  
 1   recipe_id               1055 non-null   int64  
 2   length                  1055 non-null   float64
 3   dz                      1055 non-null   float64
 4   tortuosity              1055 non-null   float64
 5   bottleneck_radius       1055 non-null   float64
 6   bottleneck_free_radius  1055 non-null   float64
 7   bottleneck_bradius      1055 non-null   float64
 8   n_layers                1055 non-null   int64  
 9   charge                  1055 non-null   int64  
 10  ionizable               1055 non-null   int64  
 11  num_positives           1055 non-null   int64  
 12  num_negatives           1055 non-null   int64  
 13  hydrophobicity          1055 non-null   float64
 14  hydropathy              1055 non-null   float64
 15

In [4]:
# 1. Dimensiones
print("Dimensiones:")
print(df.shape)

Dimensiones:
(1055, 20)


In [5]:
# 2. Tipos
print("\nTipos de variables:")
print(df.dtypes)


Tipos de variables:
frame                       int64
recipe_id                   int64
length                    float64
dz                        float64
tortuosity                float64
bottleneck_radius         float64
bottleneck_free_radius    float64
bottleneck_bradius        float64
n_layers                    int64
charge                      int64
ionizable                   int64
num_positives               int64
num_negatives               int64
hydrophobicity            float64
hydropathy                float64
polarity                  float64
logp                      float64
logd                      float64
logs                      float64
mutability                  int64
dtype: object


In [6]:
# 3. Datos faltantes
print("\nDatos faltantes:")
print(df.isna().sum())


Datos faltantes:
frame                     0
recipe_id                 0
length                    0
dz                        0
tortuosity                0
bottleneck_radius         0
bottleneck_free_radius    0
bottleneck_bradius        0
n_layers                  0
charge                    0
ionizable                 0
num_positives             0
num_negatives             0
hydrophobicity            0
hydropathy                0
polarity                  0
logp                      0
logd                      0
logs                      0
mutability                0
dtype: int64


In [7]:
# 4. Valores únicos
print("\nCantidad de valores únicos:")
print(df.nunique())


Cantidad de valores únicos:
frame                     1055
recipe_id                    1
length                    1044
dz                        1044
tortuosity                1055
bottleneck_radius          598
bottleneck_free_radius     690
bottleneck_bradius         598
n_layers                    95
charge                      15
ionizable                   21
num_positives               12
num_negatives               16
hydrophobicity              79
hydropathy                 193
polarity                   634
logp                        87
logd                       117
logs                        98
mutability                  15
dtype: int64


In [8]:
# 5. Estadísticas descriptivas
print("\nEstadísticas:")
print(df.describe().T)


Estadísticas:
                         count        mean         std         min  \
frame                   1055.0  527.000000  304.696570    0.000000   
recipe_id               1055.0    0.000000    0.000000    0.000000   
length                  1055.0  152.722837   19.918530  103.345000   
dz                      1055.0   77.270286    7.350887   60.673000   
tortuosity              1055.0    1.976032    0.166906    1.582569   
bottleneck_radius       1055.0    0.309042    0.235527   -0.209000   
bottleneck_free_radius  1055.0    0.442463    0.307349   -0.295000   
bottleneck_bradius      1055.0    0.309042    0.235527   -0.209000   
n_layers                1055.0  130.450237   18.442617   85.000000   
charge                  1055.0   -0.565877    2.318798   -8.000000   
ionizable               1055.0   11.392417    2.857717    2.000000   
num_positives           1055.0    5.413270    1.681977    1.000000   
num_negatives           1055.0    5.979147    1.985608    1.000000   
hydro

### Preguntas:
a. No, las variables no deberían analizarse todas de la misma manera. Es necesario considerar el significado y el tipo de cada variable. Las variables cuantitativas que representan características del túnel pueden resumirse mediante medidas como media, mediana, desviación estándar y rango intercuartílico. En cambio, variables como frame y recipe_id tienen principalmente una función temporal o identificatoria y no deberían interpretarse como características cuantitativas del túnel. Además, variables constantes o redundantes pueden requerir un tratamiento diferente antes de realizar análisis multivariantes.

b. Variables a expluir:
- `frame`: La excluiríamos de las variables explicativas del análisis multivariante porque representa el orden temporal de la observación. No es una propiedad del túnel.
- `recipe_id`: También probablemente la excluiríamos del análisis multivariante si funciona como identificador de configuración. Un identificador numérico no debería interpretarse como una variable cuantitativa.

c. No, la media no siempre es una buena medida descriptiva. La media puede verse afectada fuertemente por valores extremos y por distribuciones asimétricas. En esos casos, la mediana puede representar mejor el valor típico de las observaciones, ya que es menos sensible a valores extremos. Por este motivo, resulta conveniente analizar conjuntamente la media, la mediana, la desviación estándar y medidas robustas como el rango intercuartílico, además de observar la distribución de cada variable.